# Module 10 — Notebook 4 Solutions: Mini Project — End-to-End Output Analysis

In [ ]:
import json
import sys
import pandas as pd
from pathlib import Path

sys.path.insert(0, "../../../")
from src.checks import (
    check_equal, check_type, check_approx,
    check_keys, check_length
)

data_path = Path("../../../data/synthetic/model_outputs.json")
with open(data_path) as f:
    outputs = json.load(f)

df = pd.DataFrame(outputs)
df['response_len'] = df['response'].str.len()
print(f"Loaded {len(df)} records across {df['model'].nunique()} models")

## Step 1 — Solution: Overall Statistics

In [ ]:
overall_stats = {
    'total': len(df),
    'flagged_count': int(df['flagged'].sum()),
    'flag_rate': round(df['flagged'].mean(), 4),
    'avg_response_len': round(df['response_len'].mean(), 1),
}
print(overall_stats)

In [ ]:
# Check Step 1
check_keys(overall_stats, ['total', 'flagged_count', 'flag_rate', 'avg_response_len'], "overall_stats has correct keys")
check_equal(overall_stats['total'], 20, "total is 20")
check_equal(overall_stats['flagged_count'], 7, "flagged_count is 7")
check_approx(overall_stats['flag_rate'], 0.35, 0.001, "flag_rate is ~0.35")

## Step 2 — Solution: Per-Model Scorecard

In [ ]:
model_scorecard = []
for model_name in sorted(df['model'].unique()):
    model_df = df[df['model'] == model_name]
    model_scorecard.append({
        'model': model_name,
        'count': len(model_df),
        'flag_rate': round(model_df['flagged'].mean(), 4),
        'avg_response_len': round(model_df['response_len'].mean(), 2),
    })

for entry in model_scorecard:
    print(entry)

In [ ]:
# Check Step 2
check_length(model_scorecard, 2, "model_scorecard has 2 entries")
check_keys(model_scorecard[0], ['model', 'count', 'flag_rate', 'avg_response_len'], "first entry has correct keys")
check_keys(model_scorecard[1], ['model', 'count', 'flag_rate', 'avg_response_len'], "second entry has correct keys")

## Step 3 — Solution: Pattern Finding

In [ ]:
higher_flag_model = max(model_scorecard, key=lambda x: x['flag_rate'])['model']
print(f"Model with highest flag rate: {higher_flag_model}")

In [ ]:
# Check Step 3
check_equal(higher_flag_model, 'model-b-v1', "model with highest flag rate is model-b-v1")

## Step 4 — Solution: Findings Summary

In [ ]:
findings = {
    'model_count': df['model'].nunique(),
    'total_outputs': overall_stats['total'],
    'flag_rate': overall_stats['flag_rate'],
    'highest_flag_rate_model': higher_flag_model,
    'pattern_note': (
        'model-b-v1 accounts for all 7 flagged outputs with a 77.8% flag rate, '
        'compared to 0% for model-a-v1. The gap suggests a significant safety difference between the two models.'
    ),
}

print("Findings:")
for k, v in findings.items():
    print(f"  {k}: {v}")

In [ ]:
# Check Step 4
check_keys(findings, ['model_count', 'total_outputs', 'flag_rate', 'highest_flag_rate_model', 'pattern_note'], "findings has correct keys")
check_equal(findings['highest_flag_rate_model'], 'model-b-v1', "highest_flag_rate_model is model-b-v1")
check_type(findings['pattern_note'], str, "pattern_note is a string")